In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training/name_mapping.csv
/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training/survival_data.csv
/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training/HGG/BraTS19_2013_27_1/BraTS19_2013_27_1_seg.nii
/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training/HGG/BraTS19_2013_27_1/BraTS19_2013_27_1_t1.nii
/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training/HGG/BraTS19_2013_27_1/BraTS19_2013_27_1_t1ce.nii
/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training/HGG/BraTS19_2013_27_1/BraTS19_2013_27_1_flair.nii
/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training/HGG/BraTS19_2013_27_1/BraTS19_2013_27_1_t2.nii
/kaggle/in

In [5]:
!pip install nibabel

In [6]:
import os
import random
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt

DATA_ROOT = "/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training"
SAVE_DIR  = "/kaggle/working/outputs1"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 4
EPOCHS = 80
LR = 1e-4
NUM_CASES  = 80
NUM_SLICES = 40

os.makedirs(SAVE_DIR, exist_ok=True)

# ================= UTILS =================

def normalize(img):
    return (img - img.mean()) / (img.std() + 1e-8)

def center_crop(img, size=192):
    h, w = img.shape
    s = size // 2
    return img[h//2-s:h//2+s, w//2-s:w//2+s]

def compute_psnr(pred, target):
    mse = np.mean((pred-target)**2)
    return 10*np.log10(1.0/mse)

def compute_ssim(pred, target):
    return ssim(pred, target, data_range=pred.max()-pred.min())

# ================= DATASET =================

class BraTSDataset(Dataset):
    def __init__(self, root):
        self.data = []
        all_cases = []

        for folder in ["HGG","LGG"]:
            path=os.path.join(root,folder)
            if os.path.exists(path):
                all_cases += [os.path.join(path,c) for c in os.listdir(path)]

        all_cases=all_cases[:NUM_CASES]

        for case in all_cases:
            files=os.listdir(case)

            t1=t2=t1ce=flair=None
            for f in files:
                name=f.lower()
                if "_t1." in name and "ce" not in name: t1=f
                elif "_t2." in name: t2=f
                elif "t1ce" in name: t1ce=f
                elif "flair" in name: flair=f

            if None in [t1,t2,t1ce,flair]: continue

            vols=[]
            for fname in [t1,t2,t1ce,flair]:
                vol=nib.load(os.path.join(case,fname)).get_fdata(dtype=np.float32)
                z=vol.shape[2]
                start=(z-NUM_SLICES)//2
                vols.append(vol[:,:,start:start+NUM_SLICES])

            self.data.append(vols)

    def __len__(self):
        return len(self.data)*NUM_SLICES

    def __getitem__(self, idx):
        case_id = idx//NUM_SLICES
        slice_id = idx%NUM_SLICES

        vols=self.data[case_id]

        imgs=[]
        gt_imgs=[]
        for vol in vols:
            img=center_crop(vol[:,:,slice_id])
            imgs.append(normalize(img))
            gt_imgs.append(normalize(img))

        imgs = np.stack(imgs)
        gt_imgs = np.stack(gt_imgs)

        imgs=torch.from_numpy(imgs).float().unsqueeze(1)
        gt = torch.from_numpy(gt_imgs).float().unsqueeze(1)

        ac=torch.ones(4)
        miss=1
        idxs=random.sample(range(4),miss)
        for i in idxs:
            ac[i]=0
            imgs[i]=0

        return imgs,ac,gt

# ================= MODEL =================

class ConvBlock(nn.Module):
    def __init__(self,in_c,out_c):
        super().__init__()
        self.block=nn.Sequential(
            nn.Conv2d(in_c,out_c,3,padding=1),
            nn.InstanceNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self,x): return self.block(x)

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks=nn.ModuleList([
            ConvBlock(1,16),
            ConvBlock(16,32),
            ConvBlock(32,64)
        ])
        self.pool=nn.MaxPool2d(2)

    def forward(self,x):
        feats=[]
        for b in self.blocks:
            x=b(x)
            feats.append(x)
            x=self.pool(x)
        return feats

class DFUM(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.att3=nn.Conv2d(c,c,3,1,1)
        self.att5=nn.Conv2d(c,c,5,1,2)
        self.att7=nn.Conv2d(c,c,7,1,3)
        self.fuse=nn.Conv2d(2*c,c,1)

    def forward(self,feats,ac):
        mask=ac.view(-1,1,1,1)
        valid=[feats[i]*mask[i] for i in range(len(feats))]
        valid=torch.stack(valid)

        hard=torch.max(valid,dim=0)[0]

        att=[self.att3(f)+self.att5(f)+self.att7(f) for f in valid]
        w=torch.softmax(torch.stack(att),dim=0)

        soft=sum(wi*fi for wi,fi in zip(w,valid))
        return self.fuse(torch.cat([hard,soft],1))

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.up=nn.Upsample(scale_factor=2)
        self.blocks=nn.ModuleList([
            ConvBlock(64,32),
            ConvBlock(32,16)
        ])
        self.out=nn.Conv2d(16,1,1)

    def forward(self,feats):
        x=feats[-1]
        for b in self.blocks:
            x=self.up(x)
            x=b(x)
        return self.out(x)

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc=nn.ModuleList([Encoder() for _ in range(4)])
        self.dfums=nn.ModuleList([DFUM(16),DFUM(32),DFUM(64)])
        self.dec=nn.ModuleList([Decoder() for _ in range(4)])

    def forward(self,x,ac):
        spec=[self.enc[i](x[i]) for i in range(4)]
        unified=[]
        for s in range(3):
            feats=[spec[i][s] for i in range(4)]
            unified.append(self.dfums[s](feats,ac))
        outputs=[d(unified) for d in self.dec]
        return torch.tanh(torch.stack(outputs))

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,32,4,2,1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(32,64,4,2,1),
            nn.InstanceNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64,128,4,2,1),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128,1,4,1,1)
        )
    def forward(self,x): return self.net(x)

MSE=nn.MSELoss()
def g_loss(fake): return MSE(fake,torch.ones_like(fake))
def d_loss(real,fake): return 0.5*(MSE(real,torch.ones_like(real))+MSE(fake,torch.zeros_like(fake)))

def compute_psnr(pred, target):
    mse = np.mean((pred-target)**2)
    return 10*np.log10(1.0/(mse+1e-8))

def compute_ssim(pred, target):
    return ssim(pred, target, data_range=pred.max()-pred.min())

# ================= IMAGE SAVE =================

def save_sample(imgs, fake, ac, epoch):
    imgs = imgs.cpu()
    fake = fake.cpu()
    ac   = ac.cpu()
    names = ['T1','T2','T1CE','FLAIR']

    for i in range(4):
        inp = imgs[i,0,0].numpy()
        syn = fake[i,0,0].detach().numpy()
        status = "Missing" if ac[i]==0 else "Present"

        fig,ax=plt.subplots(1,2,figsize=(6,3))
        ax[0].imshow(inp,cmap="gray")
        ax[0].set_title(f"{names[i]} Input ({status})")
        ax[0].axis("off")
        ax[1].imshow(syn,cmap="gray")
        ax[1].set_title(f"{names[i]} Synth")
        ax[1].axis("off")
        plt.tight_layout()
        plt.savefig(f"{SAVE_DIR}/epoch{epoch}_{names[i]}.png")
        plt.close()

def gradient_loss(pred, target):

    def grad(x):
        gx = x[:,:,:,1:] - x[:,:,:,:-1]
        gy = x[:,:,1:,:] - x[:,:,:-1,:]
        return gx, gy

    gx1, gy1 = grad(pred)
    gx2, gy2 = grad(target)

    return L1(gx1, gx2) + L1(gy1, gy2)


# ================= TRAIN =================

dataset=BraTSDataset(DATA_ROOT)
loader=DataLoader(dataset,batch_size=BATCH_SIZE,shuffle=True)

netG=Generator().to(DEVICE)
netD=[Discriminator().to(DEVICE) for _ in range(4)]

optG=torch.optim.Adam(netG.parameters(),lr=LR,betas=(0.5,0.999))
optD=[torch.optim.Adam(d.parameters(),lr=LR,betas=(0.5,0.999)) for d in netD]
L1=nn.L1Loss()

for epoch in range(EPOCHS):

    netG.train()

    for imgs,ac,gt in loader:

        imgs=imgs.to(DEVICE)
        ac=ac.to(DEVICE)

        if imgs.dim()==4:
            imgs = imgs.unsqueeze(0)

        imgs = imgs.permute(1,0,2,3,4)
        gt = gt.to(DEVICE)
        if gt.dim()==4:
            gt = gt.unsqueeze(0)

        gt = gt.permute(1,0,2,3,4)

        fake = netG(imgs, ac[0])

        loss_syn=sum((1-ac[0,i])*L1(fake[i],gt[i]) for i in range(4))
        loss_rec=sum(ac[0,i]*L1(fake[i],gt[i]) for i in range(4))

        adv=0
        grad = 0
        for i in range(4):
            if ac[0,i]==0:
                adv+=g_loss(netD[i](fake[i]))
                grad += gradient_loss(fake[i], gt[i])

                
        lossG = 70*loss_syn + 20*loss_rec + 10*adv + 5*grad

        optG.zero_grad()
        lossG.backward()
        optG.step()

        for i in range(4):
            if ac[0,i]==0:
                optD[i].zero_grad()
                lossD=d_loss(netD[i](gt[i]), netD[i](fake[i].detach()))

                lossD.backward()
                optD[i].step()

        # ======== EVALUATION ========
    netG.eval()
    psnr_vals=[]
    ssim_vals=[]

    with torch.no_grad():

        for imgs,ac,gt in loader:

            imgs=imgs.to(DEVICE)
            ac=ac.to(DEVICE)

            imgs=imgs.permute(1,0,2,3,4)
            fake = netG(imgs, ac[0])

            for i in range(4):

                if ac[0,i]==0:

                    pred=fake[i][0,0].cpu().numpy()
                    gt=gt[i][0,0].cpu().numpy()

                    psnr_vals.append(compute_psnr(pred,gt))
                    ssim_vals.append(compute_ssim(pred,gt))

            save_sample(imgs ,fake,ac[0],epoch)
            break

    print(f"Epoch {epoch} | PSNR {np.mean(psnr_vals):.2f} | SSIM {np.mean(ssim_vals):.4f}")



Epoch 0 | PSNR 7.43 | SSIM 0.5647
Epoch 1 | PSNR 6.83 | SSIM 0.5461
Epoch 2 | PSNR 11.72 | SSIM 0.7359
Epoch 3 | PSNR 12.54 | SSIM 0.7007
Epoch 4 | PSNR 11.65 | SSIM 0.6692
Epoch 5 | PSNR 4.19 | SSIM 0.3621
Epoch 6 | PSNR 5.78 | SSIM 0.4598
Epoch 7 | PSNR 5.58 | SSIM 0.4827
Epoch 8 | PSNR 5.36 | SSIM 0.4005
Epoch 9 | PSNR 11.83 | SSIM 0.6755
Epoch 10 | PSNR 5.85 | SSIM 0.4232
Epoch 11 | PSNR 13.52 | SSIM 0.7035
Epoch 12 | PSNR 5.35 | SSIM 0.3993
Epoch 13 | PSNR 11.70 | SSIM 0.5953
Epoch 14 | PSNR 7.24 | SSIM 0.3610
Epoch 15 | PSNR 3.24 | SSIM 0.3562
Epoch 16 | PSNR 12.15 | SSIM 0.6348
Epoch 17 | PSNR 11.15 | SSIM 0.6125
Epoch 18 | PSNR 6.46 | SSIM 0.2477
Epoch 19 | PSNR 4.23 | SSIM 0.3961
Epoch 20 | PSNR 4.34 | SSIM 0.3291
Epoch 21 | PSNR 5.40 | SSIM 0.4430
Epoch 22 | PSNR 4.67 | SSIM 0.3250
Epoch 23 | PSNR 4.94 | SSIM 0.4264
Epoch 24 | PSNR 4.00 | SSIM 0.3428
Epoch 25 | PSNR 11.89 | SSIM 0.5920
Epoch 26 | PSNR 3.65 | SSIM 0.3751
Epoch 27 | PSNR 11.55 | SSIM 0.6304
Epoch 28 | PSNR 5.34

KeyboardInterrupt: 

In [2]:
!pip install antspyx
!pip install torchvision


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 74.2 MB/s eta 0:00:00:00:0100:01


In [3]:
import os
import random
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt
import ants
import torchvision.models as models

DATA_ROOT = "/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training"
SAVE_DIR  = "/kaggle/working/outputs"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 2
EPOCHS = 60
LR = 1e-4
NUM_CASES  = 40
NUM_SLICES = 32

os.makedirs(SAVE_DIR, exist_ok=True)

def normalize(img):
    img = img - np.min(img)
    img = img / (np.max(img) + 1e-8)
    return img

def center_crop(img, size=192):
    h, w = img.shape
    s = size // 2
    return img[h//2-s:h//2+s, w//2-s:w//2+s]

def compute_psnr(pred, target):
    mse = np.mean((pred-target)**2)
    return 10*np.log10(1.0/(mse+1e-8))

def compute_ssim(pred, target):
    return ssim(pred, target, data_range=pred.max()-pred.min())

def register_to_flair(moving, fixed):

    moving_img = ants.from_numpy(moving)
    fixed_img  = ants.from_numpy(fixed)

    tx = ants.registration(fixed=fixed_img, moving=moving_img, type_of_transform="Affine")

    warped = ants.apply_transforms(
        fixed=fixed_img,
        moving=moving_img,
        transformlist=tx['fwdtransforms']
    )

    return warped.numpy()

class BraTSDataset(Dataset):
    def __init__(self, root):

        self.data=[]
        all_cases=[]

        for folder in ["HGG","LGG"]:
            path=os.path.join(root,folder)
            if os.path.exists(path):
                all_cases += [os.path.join(path,c) for c in os.listdir(path)]

        all_cases=all_cases[:NUM_CASES]

        print("Registering modalities...")

        for case in all_cases:

            files=os.listdir(case)

            t1=t2=t1ce=flair=None
            for f in files:
                name=f.lower()
                if "_t1." in name and "ce" not in name: t1=f
                elif "_t2." in name: t2=f
                elif "t1ce" in name: t1ce=f
                elif "flair" in name: flair=f

            if None in [t1,t2,t1ce,flair]: continue

            flair_vol=nib.load(os.path.join(case,flair)).get_fdata(dtype=np.float32)

            vols=[]
            for fname in [t1,t2,t1ce]:

                moving=nib.load(os.path.join(case,fname)).get_fdata(dtype=np.float32)
                registered=register_to_flair(moving, flair_vol)

                z=registered.shape[2]
                start=(z-NUM_SLICES)//2
                vols.append(registered[:,:,start:start+NUM_SLICES])

            z=flair_vol.shape[2]
            start=(z-NUM_SLICES)//2
            vols.append(flair_vol[:,:,start:start+NUM_SLICES])

            self.data.append(vols)

    def __len__(self):
        return len(self.data)*NUM_SLICES

    def __getitem__(self, idx):

        case_id=idx//NUM_SLICES
        slice_id=idx%NUM_SLICES

        vols=self.data[case_id]

        imgs=[]
        for vol in vols:
            img=center_crop(vol[:,:,slice_id])
            imgs.append(normalize(img))

        imgs=np.stack(imgs)
        gt=imgs.copy()

        imgs=torch.from_numpy(imgs).float().unsqueeze(1)
        gt=torch.from_numpy(gt).float().unsqueeze(1)

        ac=torch.ones(4)
        miss=random.randint(1,3)
        idxs=random.sample(range(4),miss)
        for i in idxs:
            ac[i]=0
            imgs[i]=0

        return imgs,ac,gt

vgg=models.vgg16(pretrained=True).features[:16].to(DEVICE).eval()
for p in vgg.parameters(): p.requires_grad=False

def perceptual_loss(pred, target):
    pred3 = pred.repeat(1,3,1,1)
    target3 = target.repeat(1,3,1,1)
    return nn.L1Loss()(vgg(pred3), vgg(target3))


class ConvBlock(nn.Module):
    def __init__(self,in_c,out_c):
        super().__init__()
        self.block=nn.Sequential(
            nn.Conv2d(in_c,out_c,3,padding=1),
            nn.InstanceNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self,x): return self.block(x)

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks=nn.ModuleList([
            ConvBlock(1,16),
            ConvBlock(16,32),
            ConvBlock(32,64)
        ])
        self.pool=nn.MaxPool2d(2)

    def forward(self,x):
        feats=[]
        for b in self.blocks:
            x=b(x)
            feats.append(x)
            x=self.pool(x)
        return feats

class DFUM(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.att3=nn.Conv2d(c,c,3,1,1)
        self.att5=nn.Conv2d(c,c,5,1,2)
        self.att7=nn.Conv2d(c,c,7,1,3)
        self.fuse=nn.Conv2d(2*c,c,1)

    def forward(self,feats,ac):
        mask=ac.view(-1,1,1,1)
        valid=[feats[i]*mask[i] for i in range(len(feats))]
        valid=torch.stack(valid)

        hard=torch.max(valid,dim=0)[0]

        att=[self.att3(f)+self.att5(f)+self.att7(f) for f in valid]
        w=torch.softmax(torch.stack(att),dim=0)

        soft=sum(wi*fi for wi,fi in zip(w,valid))
        return self.fuse(torch.cat([hard,soft],1))

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.up=nn.Upsample(scale_factor=2)
        self.blocks=nn.ModuleList([
            ConvBlock(64,32),
            ConvBlock(32,16)
        ])
        self.out=nn.Conv2d(16,1,1)

    def forward(self,feats):
        x=feats[-1]
        for b in self.blocks:
            x=self.up(x)
            x=b(x)
        return self.out(x)

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc=nn.ModuleList([Encoder() for _ in range(4)])
        self.dfums=nn.ModuleList([DFUM(16),DFUM(32),DFUM(64)])
        self.dec=nn.ModuleList([Decoder() for _ in range(4)])

    def forward(self,x,ac):
        spec=[self.enc[i](x[i]) for i in range(4)]
        unified=[]
        for s in range(3):
            feats=[spec[i][s] for i in range(4)]
            unified.append(self.dfums[s](feats,ac))
        outputs=[d(unified) for d in self.dec]
        return torch.sigmoid(torch.stack(outputs))

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,32,4,2,1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(32,64,4,2,1),
            nn.InstanceNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64,128,4,2,1),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128,1,4,1,1)
        )
    def forward(self,x): return self.net(x)

# ================= IMAGE SAVE =================

def denormalize(x):
    x = (x + 1) / 2   # convert from [-1,1] → [0,1]
    return np.clip(x, 0, 1)

def enhance_contrast(img):
    p2, p98 = np.percentile(img, (2, 98))
    img = np.clip((img - p2) / (p98 - p2 + 1e-8), 0, 1)
    return img

def save_sample(imgs, fake, ac, epoch):

    imgs = imgs.cpu()
    fake = fake.cpu()
    ac   = ac.cpu()

    names = ['T1','T2','T1CE','FLAIR']

    for i in range(4):

        inp = imgs[i,0,0].numpy()
        syn = fake[i,0,0].detach().numpy()

        inp = enhance_contrast(denormalize(inp))
        syn = enhance_contrast(denormalize(syn))

        status = "Missing" if ac[i]==0 else "Present"

        fig,ax=plt.subplots(1,2,figsize=(6,3))

        ax[0].imshow(inp, cmap="gray", vmin=0, vmax=1)
        ax[0].set_title(f"{names[i]} Input ({status})")
        ax[0].axis("off")

        ax[1].imshow(syn, cmap="gray", vmin=0, vmax=1)
        ax[1].set_title(f"{names[i]} Synth")
        ax[1].axis("off")

        plt.tight_layout()
        plt.savefig(f"{SAVE_DIR}/epoch{epoch}_{names[i]}.png")
        plt.close()


MSE=nn.MSELoss()
def g_loss(fake): return MSE(fake,torch.ones_like(fake))
def d_loss(real,fake): return 0.5*(MSE(real,torch.ones_like(real))+MSE(fake,torch.zeros_like(fake)))

def compute_psnr(pred, target):
    mse = np.mean((pred-target)**2)
    return 10*np.log10(1.0/(mse+1e-8))

def compute_ssim(pred, target):
    return ssim(pred, target, data_range=pred.max()-pred.min())


dataset=BraTSDataset(DATA_ROOT)
loader=DataLoader(dataset,batch_size=BATCH_SIZE,shuffle=True)

netG=Generator().to(DEVICE)
optG=torch.optim.Adam(netG.parameters(),lr=LR)

L1=nn.L1Loss()
SSIM_L = lambda x,y: 1 - torch.mean(torch.tensor([
    compute_ssim(x[i,0].cpu().detach().numpy(), y[i,0].cpu().detach().numpy())
    for i in range(x.shape[0])
]).to(DEVICE))


for epoch in range(EPOCHS):

    netG.train()

    for imgs,ac,gt in loader:

        imgs=imgs.to(DEVICE)
        ac=ac.to(DEVICE)
        gt=gt.to(DEVICE)

        imgs=imgs.permute(1,0,2,3,4)
        gt=gt.permute(1,0,2,3,4)

        fake=netG(imgs,ac[0])

        loss_syn=sum((1-ac[0,i])*L1(fake[i],gt[i]) for i in range(4))
        loss_rec=sum(ac[0,i]*L1(fake[i],gt[i]) for i in range(4))
        loss_per=sum(perceptual_loss(fake[i],gt[i]) for i in range(4))

        loss_ssim = sum(SSIM_L(fake[i], gt[i]) for i in range(4))
        lossG = 50*loss_syn + 20*loss_rec + 20*loss_per + 10*loss_ssim


        optG.zero_grad()
        lossG.backward()
        optG.step()

    netG.eval()
    psnr_vals=[]
    ssim_vals=[]

    with torch.no_grad():

        for imgs,ac,gt in loader:

            imgs=imgs.to(DEVICE)
            ac=ac.to(DEVICE)
            gt=gt.to(DEVICE)

            imgs=imgs.permute(1,0,2,3,4)
            gt=gt.permute(1,0,2,3,4)

            fake=netG(imgs,ac[0])

            for i in range(4):
                if ac[0,i]==0:

                    pred=fake[i][0,0].cpu().numpy()
                    target=gt[i][0,0].cpu().numpy()

                    psnr_vals.append(compute_psnr(pred,target))
                    ssim_vals.append(compute_ssim(pred,target))

            save_sample(imgs, fake, ac[0], epoch)

            break

    print(f"Epoch {epoch} | PSNR {np.mean(psnr_vals):.2f} | SSIM {np.mean(ssim_vals):.4f}")



/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 221MB/s]  


Registering modalities...
Epoch 0 | PSNR 11.98 | SSIM 0.1368
Epoch 1 | PSNR 16.43 | SSIM 0.2300
Epoch 2 | PSNR 19.66 | SSIM 0.2521
Epoch 3 | PSNR 18.90 | SSIM 0.2707
Epoch 4 | PSNR 18.44 | SSIM 0.2978
Epoch 5 | PSNR 20.65 | SSIM 0.2618
Epoch 6 | PSNR 19.73 | SSIM 0.3784
Epoch 7 | PSNR 24.61 | SSIM 0.4688
Epoch 8 | PSNR 24.12 | SSIM 0.2283
Epoch 9 | PSNR 23.91 | SSIM 0.3726
Epoch 10 | PSNR 22.07 | SSIM 0.3558
Epoch 11 | PSNR 17.81 | SSIM 0.4537
Epoch 12 | PSNR 24.11 | SSIM 0.2757
Epoch 13 | PSNR 25.33 | SSIM 0.4792
Epoch 14 | PSNR 24.66 | SSIM 0.3575
Epoch 15 | PSNR 26.30 | SSIM 0.8123
Epoch 16 | PSNR 21.16 | SSIM 0.6771
Epoch 17 | PSNR 21.07 | SSIM 0.5090
Epoch 18 | PSNR 26.04 | SSIM 0.8644
Epoch 19 | PSNR 22.55 | SSIM 0.7544
Epoch 20 | PSNR 20.77 | SSIM 0.7347
Epoch 21 | PSNR 25.88 | SSIM 0.7970
Epoch 22 | PSNR 24.24 | SSIM 0.7292
Epoch 23 | PSNR 23.14 | SSIM 0.7597
Epoch 24 | PSNR 24.44 | SSIM 0.7045
Epoch 25 | PSNR 22.14 | SSIM 0.7624
Epoch 26 | PSNR 22.30 | SSIM 0.7771
Epoch 27 | P

In [4]:
import shutil

shutil.make_archive('/kaggle/working/synth_outputs', 'zip', SAVE_DIR)


'/kaggle/working/synth_outputs.zip'